## 1. Introdução e Configuração do Ambiente Estendido
Este notebook documenta o processo experimental e reprodutível de treino de um modelo de Visão por Computador baseado na arquitetura YOLO11 (especificamente a variante `yolo11s.pt`), focado na tarefa industrial de deteção de luvas de proteção em ambiente laboratorial na industria farmaceutica.

O pipeline foi desenhado para correr no ecossistema Google Colab com aceleração gráfica por hardware (GPU). A primeira fase consiste em estabelecer uma ligação persistente com o Google Drive para permitir:
1. A importação segura de scripts utilitários customizados (como o módulo de aumentos estatísticos).
2. O armazenamento centralizado e persistente da chave privada de API do Roboflow (evitando a exposição pública de credenciais).


In [ ]:
# 1. Configuração do Ambiente e Importação do Google Drive
from google.colab import drive

# Montar o sistema de ficheiros do Google Drive no caminho local do ambiente temporário (/content/drive)
drive.mount('/content/drive')

import shutil

# Definição do caminho base do projeto onde os ficheiros persistentes estão armazenados no Drive
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/gloves_project"

# Copiar o script de suporte de data augmentation customizado para a raiz do ambiente local de execuçãoshutil.copy(f"{PROJECT_DIR}/augmentation.py", "/content/augmentation.py")
shutil.copy(f"{PROJECT_DIR}/augmentation.py", "/content/augmentation.py")

# Copiar o ficheiro de texto que contém o token privado de autenticação da API do Roboflow
shutil.copy(f"{PROJECT_DIR}/api_key",         "/content/api_key")

## 2. Ecossistema de Dependências Técnicas e Ativação do TensorBoard

Para além da biblioteca nativa da Ultralytics, o projeto integra o ecossistema **Albumentations**.

O **Albumentations** é uma biblioteca de código aberto focada no processamento de imagens e na aplicação de técnicas avançadas de *Data Augmentation* (aumento artificial de dados) para Aprendizagem Profunda (*Deep Learning*).

Adicionalmente, ativamos o suporte ao **TensorBoard** através da CLI do YOLO. Isto configura um servidor local de telemetria onde podemos acompanhar visualmente as curvas de perda (*Box Loss*, *Class Loss*, *DFL Loss*) e métricas de validação por época, servindo de ferramenta de diagnóstico contra o sobreajustamento (*overfitting*).

In [ ]:
from IPython import display
display.clear_output()

import cv2
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import albumentations
import shutil

# Instalação silenciosa (-q) das dependências estruturais do projeto
!pip install ultralytics roboflow albumentations opencv-python pyyaml -q

# Configurar as definições globais da CLI do YOLO para forçar a integração automática com o TensorBoard
!yolo settings tensorboard=True

import roboflow
import ultralytics
from ultralytics import YOLO

# Executar o diagnóstico de integridade da Ultralytics para validar a GPU ativa e os recursos do sistema
ultralytics.checks()

## 3. Aquisição de Dados via Roboflow API

O conjunto de dados original foi previamente anotado, revisto e estruturado na plataforma Roboflow. Através do SDK oficial, efetuamos a autenticação automática em modo não interativo utilizando a chave copiada do Drive. O download descarrega o projeto em formato YOLOv8, o qual cria nativamente as diretorias separadas necessárias para o treino robusto: `train`, `valid` e `test`, acompanhadas pelo ficheiro descritivo de metadados `data.yaml`.

**NOTA**:É necessário ter na pasta gloves_project, na drive, um ficheiro chamado "api_key.txt" com a API key do roboflow.

In [ ]:
roboflow.login()

# Ler a chave de API de forma isolada, limpando quebras de linha com o método .strip()
with open('api_key', "r") as file:
    api_key = file.read().strip()

# Instanciar a ligação com a plataforma injetando a credencial obtida
rf = roboflow.Roboflow(api_key)

# Mapear o Workspace académico do projeto e aceder ao ID específico do repositório do trabalho
project = rf.workspace("ruis-workspace-qvcoy").project("trabalho_ia-opgfv")

# Descarregar a versão 2 estável do dataset convertida especificamente para a estrutura YOLOv8
dataset = project.version(2).download("yolov8")

## 4. Execução da Pipeline de Data Augmentation Customizada

A fim de mitigar problemas de desbalanceamento estatístico entre as classes, invocamos o script local `augmentation.py` em modo balanceado (`--mode balanced`). O algoritmo analisa a distribuição populacional das classes e aplica transformações matemáticas nas imagens sub-representadas até atingir o teto fixado de 2000 instâncias, aumentando drasticamente a capacidade de generalização do modelo final.

In [ ]:
import subprocess
from pathlib import Path

# Definir a localização local onde as pastas do dataset foram extraídas pelo Roboflow
DATASET_LOCATION = "/content/Trabalho_IA-2"

# Criar um marcador (ficheiro oculto) para garantir a idempotência do processo (não repetir aumentos desnecessariamente)
done_flag = Path(DATASET_LOCATION) / "train" / ".augmented_done"

# Executar a pipeline de data augmentation caso ainda não tenha sido processada
if done_flag.exists():
    print("Processamento de Data Augmentation ignorado: Já aplicado com sucesso anteriormente.")
    print(done_flag.read_text())
else:
    # Executar o script via subprocesso passando os parâmetros de balanceamento e meta de imagens
    result = subprocess.run(
        ["python", "augmentation.py", "--data", DATASET_LOCATION,
         "--mode", "balanced", "--target", "2000"],
        check=True
    )

## 5. Configuração Fina de Hiperparâmetros e Execução do Treino

Esta é a fase crítica do projeto. Instanciamos a rede carregando os pesos base pré-treinados do modelo `yolo11s.pt` (YOLO11 Small). Esta escolha provou ser o balanço ideal entre custo computacional (velocidade de inferência) e capacidade de extração de características em redes convolucionais.

### Justificação Científica da Escolha dos Hiperparâmetros Vencedores:
A seleção e o ajuste fino dos hiperparâmetros foram fundamentais para mitigar o enviesamento (*bias*) e garantir a convergência estocástica do modelo. A parametrização final adotada reflete a combinação empírica mais robusta para o nosso domínio de aplicação:

* **`epochs=100`**: O estabelecimento do teto macro em 100 épocas garantiu uma janela temporal computacional suficiente para superar a fase inicial de subajustamento (*underfitting*).
* **`lr0=0.001`**: A taxa de aprendizagem inicial (*Initial Learning Rate*) fixada em 0.001 revelou-se o equilíbrio ideal entre estabilidade e velocidade de convergência.
* **`patience=50`**: A parametrização do critério de paragem antecipada (*Early Stopping*) para 20 épocas atuou como o principal mecanismo de regularização dinâmica contra o sobreajustamento (*overfitting*).
* **`optimizer='SGD'`**: A especificação explícita do otimizador para **SGD** (*Stochastic Gradient Descent*) define o modelo matemático de retropropagação e atualização de pesos baseado na abordagem clássica de gradiente estocástico.


In [ ]:
from ultralytics import YOLO
from ultralytics.utils.callbacks.base import on_fit_epoch_end
import pandas as pd
from pathlib import Path
from IPython.display import clear_output, display

# Inicializar o modelo carregando a arquitetura YOLOv11 Small com pesos pré-treinados no COCO dataset
model = YOLO("yolo11s.pt")  # Load the pre-trained model weights

# Definição dos caminhos absolutos de origem (SSD local) e destino (Google Drive via Rede)
RESULTS_DIR = "/content/runs"
DRIVE_DIR   = "/content/drive/MyDrive/Colab Notebooks/gloves_project/runs"
EXPERIMENT  = "yolov11s_fase2"

def on_epoch_end(trainer):
    # ── Copiar os pesos para o Google Drive ──────────────────────────
    src_weights = Path(RESULTS_DIR) / EXPERIMENT / "weights"
    dst_weights = Path(DRIVE_DIR) / EXPERIMENT / "weights"
    dst_weights.mkdir(parents=True, exist_ok=True)
    for f in ["best.pt", "last.pt"]:
        if (src_weights / f).exists():
            shutil.copy(src_weights / f, dst_weights / f)

    # ── Visualização inline das curvas de treino ──────────────────────────────
    csv_path = Path(RESULTS_DIR) / EXPERIMENT / "results.csv"
    if not csv_path.exists():
        return
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    clear_output(wait=True)
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle(f"Epoch {trainer.epoch + 1}/{trainer.epochs}")
    axes[0,0].plot(df["epoch"], df["train/box_loss"], label="train", marker='o')
    axes[0,0].plot(df["epoch"], df["val/box_loss"],   label="val",   marker='o')
    axes[0,0].set_title("Box Loss"); axes[0,0].legend()
    axes[0,1].plot(df["epoch"], df["train/cls_loss"], label="train", marker='o')
    axes[0,1].plot(df["epoch"], df["val/cls_loss"],   label="val",   marker='o')
    axes[0,1].set_title("Class Loss"); axes[0,1].legend()
    axes[0,2].plot(df["epoch"], df["train/dfl_loss"], label="train", marker='o')
    axes[0,2].plot(df["epoch"], df["val/dfl_loss"],   label="val",   marker='o')
    axes[0,2].set_title("DFL Loss"); axes[0,2].legend()
    axes[1,0].plot(df["epoch"], df["metrics/mAP50(B)"],    marker='o')
    axes[1,0].set_title("mAP@0.5")
    axes[1,1].plot(df["epoch"], df["metrics/recall(B)"],   marker='o')
    axes[1,1].set_title("Recall")
    axes[1,2].plot(df["epoch"], df["metrics/precision(B)"], marker='o')
    axes[1,2].set_title("Precision")
    plt.tight_layout()
    display(fig)
    plt.close()

def on_train_end(trainer):
    # Mapear as diretorias locais e remotas usando a biblioteca orientada a objetos Path
    src = Path(RESULTS_DIR) / EXPERIMENT
    dst = Path(DRIVE_DIR) / EXPERIMENT

    # Executar a cópia recursiva da árvore de diretórios.
    # O parâmetro dirs_exist_ok=True previne exceções caso a pasta de destino já tenha sido criada
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"[callback] Training complete — full results copied to Drive.")

# Injetar os callbacks customizados no ciclo de vida do modelo
model.add_callback("on_fit_epoch_end", on_epoch_end)
model.add_callback("on_train_end", on_train_end)

# ─────────────────────────────────────────────────────────────────────────────
# model.train() — # Bloco de execução do treino com a parametrização explícita do dicionário da Ultralytics
# ─────────────────────────────────────────────────────────────────────────────
results = model.train(

    # ── Ficheiros de Configuração e Dados ──────────────────────────────────────
    data="/content/Trabalho_IA-2/data.yaml",

    # ── Configurações principais do treino ────────────────────────────────────
    epochs=100,         # Número total de épocas de treino.
                        # Mais épocas = mais aprendizagem, mas maior risco de overfitting.
                        # Intervalo típico: 50–300.

    patience=50,        # Early stopping: interrompe o treino se não houver melhoria
                        # durante este número de épocas. Definir como 0 para desativar.

    batch=16,           # Número de imagens por batch de treino.
                        # Valores maiores = mais rápido, mas requer mais VRAM. Usar -1 para AutoBatch.

    imgsz=640,          # Tamanho da imagem de entrada (píxeis). As imagens são redimensionadas para este quadrado.
                        # Valores comuns: 416, 512, 640, 1280.

    # ── Hardware ──────────────────────────────────────────────────────────────
    device=0,       # Dispositivo de treino.
                        # "cpu"       → apenas CPU (lento)
                        # 0           → primeira GPU CUDA
                        # [0, 1]      → multi-GPU (CUDA)
                        # "mps"       → GPU Apple Silicon

    workers=8,          # Número de threads de DataLoader para carregamento de imagens.
                        # Reduzir em caso de erros de memória ou bottlenecks de CPU.

    # ── Checkpointing & output ────────────────────────────────────────────────
    project=RESULTS_DIR,   # Pasta raiz onde os resultados do treino são guardados.
    name="yolov11s_fase2",   # Nome da sub-pasta para esta execução específica.
                             # Os resultados são guardados em <project>/<name>/.

    exist_ok=False,     # Se False, é criada uma nova pasta numerada quando o nome já existe.
                        # Se True, a pasta existente é sobrescrita.

    save=True,          # Guardar checkpoints (best.pt e last.pt) durante o treino.
    save_period=-1,     # Guardar um checkpoint a cada N épocas. -1 = desativado.
                        # Exemplo: save_period=10 guarda a cada 10 épocas.

    # ── Transfer learning ─────────────────────────────────────────────────────
    pretrained=True,    # Iniciar a partir de pesos pré-treinados (recomendado).
                        # Definir como False para treinar do zero.

    freeze=0,           # Congelar as primeiras N camadas do backbone (não atualizar os seus pesos).
                        # Útil para fine-tuning.
                        # Exemplo: freeze=10 congela as primeiras 10 camadas.

    # ── Otimizador ─────────────────────────────────────────────────────────────
    optimizer="SGD",    # Algoritmo de otimização. Opções:
                        # "SGD", "Adam", "AdamW", "NAdam", "RAdam", "RMSProp", "auto"
                        # "auto" seleciona SGD ou AdamW com base no modelo.

    lr0=0.001,          # Taxa de aprendizagem inicial.
                        # Valores mais baixos (ex: 0.001) são mais seguros para fine-tuning.

    lrf=0.01,           # Taxa de aprendizagem final como fração de lr0.
                        # O scheduler decai lr0 → lr0 * lrf ao longo do treino.

    momentum=0.937,     # Momentum do SGD / beta1 do Adam.
                        # Controla a influência dos gradientes passados na atualização atual.

    weight_decay=0.0005, # Penalização L2 de regularização — desencoraja pesos elevados
                         # e ajuda a prevenir overfitting.

    warmup_epochs=3.0,  # Número de épocas de aquecimento da taxa de aprendizagem no início.
                        # A LR sobe gradualmente de 0 até lr0 durante esta fase.

    warmup_momentum=0.8, # Momentum inicial durante a fase de aquecimento.
    warmup_bias_lr=0.1,  # Taxa de aprendizagem inicial para os parâmetros de bias durante o aquecimento.

    # ── Loss weights ──────────────────────────────────────────────────────────
    box=7.5,            # Peso da perda de regressão das bounding boxes.
                        # Aumentar para fazer o modelo focar mais na precisão das caixas.

    cls=0.5,            # Peso da perda de classificação.
    dfl=1.5,            # Peso da Distribution Focal Loss (refinamento das caixas).

    # ── Augmentation ──────────────────────────────────────────────────────────
    hsv_h=0.015,        # Variação aleatória de matiz (fração de 360°). Adiciona variedade de cor.
    hsv_s=0.7,          # Variação aleatória de saturação. Intervalo: 0.0–1.0.
    hsv_v=0.1,          # Variação aleatória de brilho (valor). Intervalo: 0.0–1.0. [* Alterado para 0.1]

    degrees=0.0,        # Intervalo de rotação aleatória em graus (ex: 10 → ±10°).
    translate=0.1,      # Translação aleatória como fração do tamanho da imagem (ex: 0.1 = 10%).
    scale=0.5,          # Fator de escala aleatório (ex: 0.5 significa zoom entre 50%–150%).
    shear=0.0,          # Ângulo de corte aleatório em graus.
    perspective=0.0,    # Distorção de perspetiva aleatória. Intervalo: 0.0–0.001.
    flipud=0.0,         # Probabilidade de flip vertical. 0.0 = desativado.
    fliplr=0.0,         # Probabilidade de flip horizontal. 0.5 = 50% de probabilidade por imagem. [* Alterado para 0.0]

    mosaic=0.5,         # Probabilidade de augmentation Mosaic (combina 4 imagens).
                        # Muito eficaz para deteção de objetos pequenos. 0.0 = desativado.

    mixup=0.0,          # Probabilidade de augmentation MixUp (mistura 2 imagens).
                        # Útil para melhorar a generalização.

    copy_paste=0.0,     # Probabilidade de augmentation Copy-Paste (cola objetos de uma
                        # imagem noutra). Bom para segmentação de instâncias.

    # ── Avaliação ────────────────────────────────────────────────────────────
    val=True,           # Executar validação após cada época para monitorizar mAP, loss, etc.

    plots=True,         # Gerar e guardar gráficos de treino (curvas de perda, matriz de
                        # confusão, curva PR) na pasta de resultados.

    # ── Reprodutibilidade ───────────────────────────────────────────────────────
    seed=0,             # Semente aleatória para reprodutibilidade. Usar a mesma semente
                        # para obter resultados idênticos entre execuções.

    deterministic=True, # Forçar operações CUDA deterministas.
                        # Garante reprodutibilidade mas pode abrandar ligeiramente o treino.

    # ── Outros ──────────────────────────────────────────────────────────────────
    resume=False,       # Retomar o treino a partir do último checkpoint guardado (last.pt).
                        # Definir como o caminho do checkpoint para retomar uma execução específica.

    amp=True,           # Precisão Mista Automática (FP16). Acelera o treino e
                        # reduz o uso de VRAM em hardware compatível.

    fraction=1.0,       # Fração do dataset de treino a utilizar.
                        # Exemplo: 0.1 usa apenas 10% das imagens (útil para testes rápidos).

    profile=False,      # Perfilar velocidades ONNX e TensorRT durante o treino.
                        # Útil para benchmarking de alvos de deployment.

    verbose=True,       # Imprimir logs detalhados do treino na consola.
)